In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.font_manager import FontProperties, fontManager

# Figure 2: Temporal patterns, 2016–2025

# Repository root
ROOT_DIR = Path.cwd()

if ROOT_DIR.name == "figure_scripts":
    ROOT_DIR = ROOT_DIR.parent

# Input data
DATA_FILE = (
    ROOT_DIR
    / "dataset"
    / "data"
    / "processed"
    / "agroclimate_with_indicators.parquet"
)

# Output directory
OUTPUT_DIR = ROOT_DIR / "figure_scripts" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Configure font
FONT_FILE = ROOT_DIR / "calibri.ttf"

if FONT_FILE.exists():
    fontManager.addfont(str(FONT_FILE))
    calibri = FontProperties(fname=str(FONT_FILE))
    plt.rcParams["font.family"] = calibri.get_name()
else:
    calibri = FontProperties(family="DejaVu Sans")
    plt.rcParams["font.family"] = "DejaVu Sans"

plt.rcParams["font.size"] = 16

data = pd.read_parquet(
    DATA_FILE,
    columns=[
        "grid_id",
        "date",
        "temperature_2m_mean",
        "precipitation_sum",
        "et0_fao_evapotranspiration",
    ],
)

data["date"] = pd.to_datetime(data["date"])
data["month"] = data["date"].dt.to_period("M").dt.to_timestamp()

monthly_grid = (
    data.groupby(["grid_id", "month"], as_index=False)
    .agg(
        temperature_2m_mean=("temperature_2m_mean", "mean"),
        precipitation_sum=("precipitation_sum", "sum"),
        et0_fao_evapotranspiration=("et0_fao_evapotranspiration", "sum"),
    )
)

monthly = (
    monthly_grid.groupby("month", as_index=False)
    .agg(
        temperature_2m_mean=("temperature_2m_mean", "mean"),
        precipitation_sum=("precipitation_sum", "mean"),
        et0_fao_evapotranspiration=("et0_fao_evapotranspiration", "mean"),
    )
)

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

axes[0].plot(monthly["month"], monthly["temperature_2m_mean"])
axes[0].set_title(
    "Monthly Mean Temperature",
    fontsize=16,
    fontproperties=calibri,
    pad=12
)
axes[0].set_ylabel(
    "Temperature (°C)",
    fontsize=16,
    fontproperties=calibri
)

axes[1].plot(monthly["month"], monthly["precipitation_sum"])
axes[1].set_title(
    "Monthly Precipitation",
    fontsize=16,
    fontproperties=calibri,
    pad=12
)
axes[1].set_ylabel(
    "Precipitation (mm)",
    fontsize=16,
    fontproperties=calibri
)

axes[2].plot(monthly["month"], monthly["et0_fao_evapotranspiration"])
axes[2].set_title(
    "Monthly Reference Evapotranspiration",
    fontsize=16,
    fontproperties=calibri,
    pad=12
)
axes[2].set_ylabel(
    "ET₀ (mm)",
    fontsize=16,
    fontproperties=calibri
)
axes[2].set_xlabel(
    "Month",
    fontsize=16,
    fontproperties=calibri
)

for ax in axes:
    ax.tick_params(axis="both", labelsize=14)
    ax.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()

# Save figure in PNG and PDF formats
plt.savefig(
    OUTPUT_DIR / "figure1_temporal_patterns.png",
    dpi=300,
    bbox_inches="tight"
)
plt.savefig(
    OUTPUT_DIR / "figure1_temporal_patterns.pdf",
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print("Temporal patterns figure saved:")
print(OUTPUT_DIR / "figure1_temporal_patterns.png")
print(OUTPUT_DIR / "figure1_temporal_patterns.pdf")